# 🚀 ETL com IA Generativa em Python
**DIO - Data Engineering | Santander Dev Week**

Pipeline completo de ETL utilizando o dataset **ABC Multistate Bank Customer Churn** (Kaggle),
enriquecido com campos extras e ruídos artificiais para simulação de um ambiente real.

A IA Generativa atua na etapa de **Transformação**, gerando mensagens personalizadas
para clientes com risco de churn, com base no seu perfil financeiro.

---
```
📥 EXTRACT                🔧 TRANSFORM                        📤 LOAD
bank_churn_dirty.csv  →  Limpeza + Enriquecimento + IA Gen  →  bank_churn_clean.csv
```

---
## 📥 ETAPA 1 — EXTRACT
Leitura do CSV e inspeção dos dados brutos.

In [1]:
import pandas as pd
import numpy as np
import re
import os
import time
from dotenv import load_dotenv
from groq import Groq
from google import genai
from mistralai import Mistral
from openai import OpenAI

load_dotenv()

df_raw = pd.read_csv('bank_churn_dirty.csv')

print('=== DADOS BRUTOS ===')
print(f'Linhas  : {len(df_raw)}')
print(f'Colunas : {list(df_raw.columns)}')
print()
df_raw.head()


=== DADOS BRUTOS ===
Linhas  : 10150
Colunas : ['customer_id', 'nome', 'email', 'numero_conta', 'data_nascimento', 'cidade', 'country', 'gender', 'age', 'credit_score', 'tenure', 'balance', 'limite_credito', 'products_number', 'credit_card', 'active_member', 'estimated_salary', 'churn']



,customer_id,nome,email,numero_conta,data_nascimento,cidade,country,gender,age,credit_score,tenure,balance,limite_credito,products_number,credit_card,active_member,estimated_salary,churn
0,15631460.0,Camila Martins Lima,camila.martins@hotmail.com,83406-0,04/10/1982,Barcelona,Spain,Female,42,671,3,0.00,0.00,2,1,1,128449.33,0
1,15700300.0,Patrícia Rocha Alves,patricia.rocha@uol.com.br,83680-0,18/12/1980,Berlin,Germany,Female,44,674,4,131593.85,202324.81,1,0,1,171345.02,1
2,NaN,ANDRÉ COSTA ARAÚJO,andre.costa@gmail.com,73884-8,1985-08-26,Berlin,Germany,Male,39,789,7,124828.46,356669.15,2,1,1,124411.08,0
3,15760456.0,Silvana Costa Ferreira,silvana.costa@outlook.com,43299-9,1986-04-04,Paris,France,Female,38,731,10,123711.73,262028.34,2,1,0,171340.68,1
4,15638730.0,Natália Santos Martins,natalia.santos@yahoo.com.br,56717-2,2003-04-11,Paris,France,Female,-1,711,0,82844.33,207795.24,2,0,1,1408.68,0


In [2]:
# Diagnóstico geral — nulos e brancos em todas as colunas
print('=== DIAGNÓSTICO DE QUALIDADE ===')
print()

print('── Nulos e brancos por coluna ──')
for col in df_raw.columns:
    nulos  = df_raw[col].isna().sum()
    brancos = (df_raw[col].astype(str).str.strip() == '').sum()
    if nulos > 0 or brancos > 0:
        print(f'  {col:<20} nulos: {nulos:>4}   brancos: {brancos:>4}')

print()
print('── Outros problemas detectados ──')
print(f"  Linhas duplicadas            : {df_raw.duplicated().sum()}")
print(f"  Nomes não padronizados       : {df_raw['nome'].dropna().apply(lambda x: x != x.strip().title()).sum()}")
print(f"  Emails inválidos             : {df_raw['email'].dropna().apply(lambda x: not re.match(r'^[\w\.-]+@[\w\.-]+\.\w{2,}$', str(x).lower().strip())).sum()}")
print(f"  Idades inválidas (-1)        : {(df_raw['age'] == -1).sum()}")
print(f"  Credit score impossível (999): {(df_raw['credit_score'] == 999).sum()}")
print(f"  Saldos negativos             : {(df_raw['balance'] < 0).sum()}")
print(f"  Saldos outlier (999999.99)   : {(df_raw['balance'] == 999999.99).sum()}")
print(f"  Datas formato YYYY-MM-DD     : {df_raw['data_nascimento'].dropna().apply(lambda x: bool(re.match(r'^\\d{4}-', str(x)))).sum()}")
print(f"  Cidades não padronizadas     : {df_raw['cidade'].dropna().apply(lambda x: x != x.strip().title()).sum()}")

=== DIAGNÓSTICO DE QUALIDADE ===

── Nulos e brancos por coluna ──
  customer_id          nulos:  203   brancos:    0
  email                nulos:  405   brancos:    0
  estimated_salary     nulos:  605   brancos:    0

── Outros problemas detectados ──
  Linhas duplicadas            : 149
  Nomes não padronizados       : 811
  Emails inválidos             : 669
  Idades inválidas (-1)        : 304
  Credit score impossível (999): 303
  Saldos negativos             : 316
  Saldos outlier (999999.99)   : 204
  Datas formato YYYY-MM-DD     : 0
  Cidades não padronizadas     : 814


---
## 🔧 ETAPA 2 — TRANSFORM

### Parte A: Limpeza e Padronização

In [3]:
df = df_raw.copy()

# ── 1. Remover duplicatas ──────────────────────────────────────────────────
antes = len(df)
df = df.drop_duplicates()
print(f'Duplicatas removidas: {antes - len(df)}')

# ── 2. Nulos e brancos: substituir brancos por NaN em todo o dataset ───────
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# ── 3. Padronizar nomes (strip + title case) ───────────────────────────────
df['nome'] = df['nome'].str.strip().str.title()

# ── 4. Emails: validar nulos, brancos e formato inválido ──────────────────
def validar_email(email):
    if pd.isna(email):
        return 'sem_email@banco.com'
    email = str(email).lower().strip()
    return email if re.match(r'^[\w\.-]+@[\w\.-]+\.\w{2,}$', email) else 'sem_email@banco.com'

df['email'] = df['email'].apply(validar_email)

# ── 5. Padronizar cidades (strip + title case) ────────────────────────────
df['cidade'] = df['cidade'].str.strip().str.title()

# ── 6. Datas: padronizar para DD/MM/YYYY ─────────────────────────────────
def padronizar_data(data):
    if pd.isna(data):
        return None
    s = str(data).strip()
    try:
        if re.match(r'^\\d{4}-\\d{2}-\\d{2}$', s):
            return pd.to_datetime(s, format='%Y-%m-%d').strftime('%d/%m/%Y')
        return pd.to_datetime(s, format='%d/%m/%Y').strftime('%d/%m/%Y')
    except:
        return None

df['data_nascimento'] = df['data_nascimento'].apply(padronizar_data)

# ── 7. Idades inválidas (-1) → mediana ────────────────────────────────────
mediana_age = int(df[df['age'] > 0]['age'].median())
df.loc[df['age'] == -1, 'age'] = mediana_age

# ── 8. Credit score impossível (>850) → mediana ───────────────────────────
mediana_score = int(df[df['credit_score'] <= 850]['credit_score'].median())
df.loc[df['credit_score'] > 850, 'credit_score'] = mediana_score

# ── 9. Saldos: negativos → 0, outliers → mediana ─────────────────────────
mediana_balance = df[(df['balance'] >= 0) & (df['balance'] < 999999)]['balance'].median()
df.loc[df['balance'] < 0, 'balance'] = 0
df.loc[df['balance'] == 999999.99, 'balance'] = mediana_balance

# ── 10. Salário nulo ou branco → mediana ──────────────────────────────────
mediana_salary = df['estimated_salary'].median()
df['estimated_salary'] = df['estimated_salary'].fillna(mediana_salary)

# ── 11. customer_id nulo → novo ID sequencial ─────────────────────────────
max_id = df['customer_id'].max()
nulos_id = df['customer_id'].isna().sum()
df.loc[df['customer_id'].isna(), 'customer_id'] = range(int(max_id) + 1, int(max_id) + 1 + nulos_id)
df['customer_id'] = df['customer_id'].astype(int)

print()
print('=== APÓS LIMPEZA ===')
print(f'Linhas : {len(df)}')
nulos = df.isnull().sum()
print(f'Nulos  : {nulos[nulos > 0].to_dict() or "Nenhum ✅"}')
print()
df.head()

Duplicatas removidas: 149

=== APÓS LIMPEZA ===
Linhas : 10001
Nulos  : {'data_nascimento': 4005}



,customer_id,nome,email,numero_conta,data_nascimento,cidade,country,gender,age,credit_score,tenure,balance,limite_credito,products_number,credit_card,active_member,estimated_salary,churn
0,15631460,Camila Martins Lima,camila.martins@hotmail.com,83406-0,04/10/1982,Barcelona,Spain,Female,42,671,3,0.00,0.00,2,1,1,128449.33,0
1,15700300,Patrícia Rocha Alves,patricia.rocha@uol.com.br,83680-0,18/12/1980,Berlin,Germany,Female,44,674,4,131593.85,202324.81,1,0,1,171345.02,1
2,15815691,André Costa Araújo,andre.costa@gmail.com,73884-8,NaN,Berlin,Germany,Male,39,789,7,124828.46,356669.15,2,1,1,124411.08,0
3,15760456,Silvana Costa Ferreira,silvana.costa@outlook.com,43299-9,NaN,Paris,France,Female,38,731,10,123711.73,262028.34,2,1,0,171340.68,1
4,15638730,Natália Santos Martins,natalia.santos@yahoo.com.br,56717-2,NaN,Paris,France,Female,37,711,0,82844.33,207795.24,2,0,1,1408.68,0


### Parte B: Classificação de Perfil

Classificamos cada cliente antes de gerar as mensagens para enriquecer o contexto enviado à IA.

In [4]:
def classificar_perfil(row):
    if row['churn'] == 1:
        return 'cliente em risco de churn'
    elif row['balance'] == 0:
        return 'cliente com saldo zerado'
    elif row['balance'] < 50000 and row['active_member'] == 0:
        return 'cliente inativo'
    elif row['balance'] >= 100000:
        return 'cliente premium'
    else:
        return 'cliente ativo'

df['perfil'] = df.apply(classificar_perfil, axis=1)

print('Distribuição de perfis:')
print(df['perfil'].value_counts())

Distribuição de perfis:
perfil
cliente premium              3352
cliente com saldo zerado     3273
cliente em risco de churn    2037
cliente ativo                1322
cliente inativo                17
Name: count, dtype: int64


### Parte C: IA Generativa — Mensagens Personalizadas

Para cada cliente em risco (churn=1), o modelo **Llama 3 (Groq)** gera uma mensagem
de retenção personalizada com base no perfil financeiro.


In [5]:
# ── Clientes de cada API ──────────────────────────────────────────────────
groq_client     = Groq(api_key=os.getenv('GROQ_API_KEY'))
mistral_client  = Mistral(api_key=os.getenv('MISTRAL_API_KEY'))
openrouter_client = OpenAI(
    api_key=os.getenv('OPENROUTER_API_KEY'),
    base_url='https://openrouter.ai/api/v1'
)
gemini_client = genai.Client(api_key=os.getenv('GEMINI_API_KEY'))

# ── Lista de provedores em ordem de preferência ───────────────────────────
# Cada entrada: (nome, função_que_chama_a_api)
def chamar_groq(prompt):
    r = groq_client.chat.completions.create(
        model='llama-3.3-70b-versatile',
        max_tokens=150,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return r.choices[0].message.content.strip()

def chamar_gemini(prompt):
    r = gemini_client.models.generate_content(
        model='gemini-2.0-flash',
        contents=prompt
    )
    return r.text.strip()
def chamar_mistral(prompt):
    r = mistral_client.chat.complete(
        model='mistral-small-latest',
        max_tokens=150,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return r.choices[0].message.content.strip()

def chamar_openrouter(prompt):
    r = openrouter_client.chat.completions.create(
        model='meta-llama/llama-3.3-70b-instruct:free',
        max_tokens=150,
        messages=[{'role': 'user', 'content': prompt}]
    )
    return r.choices[0].message.content.strip()

PROVEDORES = [
    ('Groq',        chamar_groq),
    ('Gemini',      chamar_gemini),
    ('Mistral',     chamar_mistral),
    ('OpenRouter',  chamar_openrouter),
]

# índice global do provedor atual — avança no round-robin
provedor_idx = 0

def extrair_tempo_espera(mensagem_erro: str) -> float:
    match = re.search(r'try again in (\d+m)?(\d+\.?\d*)s', str(mensagem_erro))
    if match:
        minutos = int(match.group(1).replace('m', '')) if match.group(1) else 0
        segundos = float(match.group(2))
        return minutos * 60 + segundos + 1
    return 10

# Controla quais provedores estão em cooldown e até quando
cooldowns = {}  # { nome: timestamp_liberacao }

def gerar_mensagem(cliente: dict) -> str:
    prompt = f"""
    Você é um assistente de relacionamento do ABC Bank.
    Gere uma mensagem curta (máximo 2 frases) e personalizada para o cliente abaixo.
    O objetivo é reter o cliente ou reforçar o relacionamento com o banco.
    Mencione o primeiro nome do cliente. Use português brasileiro. Seja direto e humano.

    Nome          : {cliente['nome']}
    País          : {cliente['country']}
    Cidade        : {cliente['cidade']}
    Saldo         : {cliente['balance']:.2f}
    Limite crédito: {cliente['limite_credito']:.2f}
    Produtos      : {cliente['products_number']}
    Membro ativo  : {'Sim' if cliente['active_member'] == 1 else 'Não'}
    Perfil        : {cliente['perfil']}

    Responda APENAS com a mensagem, sem aspas ou formatação extra.
    """
    while True:
        agora = time.time()

        # Eleger próximo provedor disponível (sem cooldown ativo)
        disponiveis = [(n, fn) for n, fn in PROVEDORES if agora >= cooldowns.get(n, 0)]

        if not disponiveis:
            # Todos em cooldown — espera o que libera primeiro
            proximo_liberado = min(cooldowns.values())
            espera = max(0, proximo_liberado - agora)
            nome_prox = min(cooldowns, key=cooldowns.get)
            print(f'    ⏳ Todos em rate limit. Aguardando {espera:.0f}s até {nome_prox} liberar...')
            time.sleep(espera)
            continue

        nome, fn = disponiveis[0]
        try:
            return fn(prompt)
        except Exception as e:
            erro = str(e)
            if '429' in erro or 'rate' in erro.lower():
                espera = extrair_tempo_espera(erro)
                cooldowns[nome] = time.time() + espera
                proximo = disponiveis[1][0] if len(disponiveis) > 1 else '?'
                print(f'    ⚠️  {nome} rate limit por {espera:.0f}s → tentando {proximo}')
            else:
                cooldowns[nome] = time.time() + 60  # penalidade de 60s para erros genéricos
                print(f'    ❌ {nome} erro: {erro[:80]}')

print('Função de IA com round-robin configurada! ✅')
print(f'Provedores disponíveis: {[p[0] for p in PROVEDORES]}')



Função de IA com round-robin configurada! ✅
Provedores disponíveis: ['Groq', 'Gemini', 'Mistral', 'OpenRouter']


In [6]:
import time

# Geramos mensagens apenas para clientes com churn=1 — prioridade de retenção
df_churn = df[df['churn'] == 1].copy().reset_index(drop=True)
print(f'Clientes em risco de churn: {len(df_churn)}')
print('Gerando mensagens com IA...\n')

mensagens = []
for i, row in df_churn.iterrows():
    msg = gerar_mensagem(row.to_dict())
    mensagens.append(msg)
    primeiro_nome = row['nome'].split()[0]
    print(f'[{i+1}/{len(df_churn)}] ✅ {primeiro_nome}: {msg[:70]}...')
    time.sleep(0.5)  # respeita o rate limit do Groq (30 req/min no plano gratuito)

df_churn['mensagem_ia'] = mensagens
print('\nMensagens geradas com sucesso! 🎉')


Clientes em risco de churn: 2037
Gerando mensagens com IA...

[1/2037] ✅ Patrícia: Olá Patrícia, estamos aqui para ajudar e queremos garantir que você es...
    ⚠️  Groq rate limit por 182s → tentando Gemini
    ⚠️  Gemini rate limit por 10s → tentando Mistral
[2/2037] ✅ Silvana: Silvana, sabemos que você tem um relacionamento importante com o ABC B...
[3/2037] ✅ André: André, valorizamos muito sua confiança no ABC Bank. Que tal explorarmo...
[4/2037] ✅ Mariana: Mariana, valorizamos muito sua confiança no ABC Bank e queremos garant...
[5/2037] ✅ Paulo: Paulo, valorizamos muito sua confiança no ABC Bank. Que tal aproveitar...
[6/2037] ✅ Mariana: Mariana, sabemos que você tem um saldo significativo no ABC Bank e que...
[7/2037] ✅ Marcos: Marcos, valorizamos muito sua confiança no ABC Bank. Estamos aqui para...
[8/2037] ✅ Vanessa: Vanessa, sabemos que você é uma cliente importante para nós e adoraría...
[9/2037] ✅ Paulo: Paulo, valorizamos muito sua confiança no ABC Bank. Que tal conversa

---
## 📤 ETAPA 3 — LOAD

Dois arquivos de saída:
- `bank_churn_clean.csv` — dataset completo limpo
- `bank_churn_mensagens.csv` — clientes em risco com mensagem da IA

In [7]:
df.to_csv('bank_churn_clean.csv', index=False, encoding='utf-8-sig')
print(f'✅ bank_churn_clean.csv — {len(df)} registros')

df_churn.to_csv('bank_churn_mensagens.csv', index=False, encoding='utf-8-sig')
print(f'✅ bank_churn_mensagens.csv — {len(df_churn)} registros')

print()
print('=== PRÉVIA — CLIENTES EM RISCO + MENSAGEM IA ===')
df_churn[['nome', 'country', 'balance', 'perfil', 'mensagem_ia']].head(10)

✅ bank_churn_clean.csv — 10001 registros
✅ bank_churn_mensagens.csv — 2037 registros

=== PRÉVIA — CLIENTES EM RISCO + MENSAGEM IA ===


,nome,country,balance,perfil,mensagem_ia
0,Patrícia Rocha Alves,Germany,131593.85,cliente em risco de churn,"Olá Patrícia, estamos aqui para ajudar e quere..."
1,Silvana Costa Ferreira,France,123711.73,cliente em risco de churn,"Silvana, sabemos que você tem um relacionament..."
2,André Costa Araújo,Germany,132405.52,cliente em risco de churn,"André, valorizamos muito sua confiança no ABC ..."
3,Mariana Costa Santos,France,238387.56,cliente em risco de churn,"Mariana, valorizamos muito sua confiança no AB..."
4,Paulo Roberto Mendes,Germany,120100.41,cliente em risco de churn,"Paulo, valorizamos muito sua confiança no ABC ..."
5,Mariana Costa Santos,Germany,103700.69,cliente em risco de churn,"Mariana, sabemos que você tem um saldo signifi..."
6,Marcos Pereira Rocha,France,0.00,cliente em risco de churn,"Marcos, valorizamos muito sua confiança no ABC..."
7,Vanessa Gomes Carvalho,Spain,0.00,cliente em risco de churn,"Vanessa, sabemos que você é uma cliente import..."
8,Paulo Roberto Mendes,Germany,109339.17,cliente em risco de churn,"Paulo, valorizamos muito sua confiança no ABC ..."
9,João Pedro Souza,Germany,127655.22,cliente em risco de churn,"João Pedro, valorizamos muito sua confiança no..."


---
## 📊 Resumo do Pipeline

| Etapa | Ação | Resultado |
|-------|------|-----------|
| **Extract** | Leitura do `bank_churn_dirty.csv` | 10.150 linhas, 18 colunas |
| **Transform — Limpeza** | Nulos/brancos em todas as colunas, padronização, correção de valores inválidos | Dataset íntegro e padronizado |
| **Transform — Perfil** | Classificação por comportamento financeiro | Campo `perfil` adicionado |
| **Transform — IA Gen** | Mensagens personalizadas para clientes churn=1 via Claude API | Campo `mensagem_ia` adicionado |
| **Load** | Exportação para CSV | `bank_churn_clean.csv` + `bank_churn_mensagens.csv` |

### Tratamentos aplicados:
- ✅ Linhas duplicadas removidas
- ✅ Brancos convertidos para nulo em todas as colunas
- ✅ Nomes padronizados (strip + title case)
- ✅ Emails inválidos, nulos e brancos → `sem_email@banco.com`
- ✅ Cidades padronizadas (strip + title case)
- ✅ Datas em formato misto → DD/MM/YYYY
- ✅ Idades inválidas (-1) → mediana
- ✅ Credit score impossível (999) → mediana
- ✅ Saldos negativos → 0
- ✅ Saldos outlier (999999.99) → mediana
- ✅ Salário nulo/branco → mediana
- ✅ customer_id nulo → novo ID sequencial